# 托管你的智能体

你已经在[笔记本 00](./00_The_one_liner_research_agent.ipynb) 中构建了一个研究智能体。它运行在你的笔记本电脑上。现在别人也需要使用它：一位同事、一个 cron 任务、一个 Web 应用、一位客户。这意味着它必须*在你终端之外的某个地方*运行，保持在线，在重启之间延续对话，并且不泄露你的 API 密钥。

本笔记本把完全相同的智能体通过三个层级进行部署：

| 层级 | 运行位置 | 适用场景 |
|---|---|---|
| **1. Docker** | 你的机器 / 单台虚拟机 | 开发循环、内部工具、单租户 |
| **2. Modal** | 托管的无服务器 | 你想要一个 URL 和缩零伸缩，而不必管理基础设施 |
| **3. Kubernetes** | 你自己的集群 | 多租户、受监管环境、完全掌控 |

三个层级的智能体代码、容器镜像和 HTTP 接口**完全相同**。只有容器外围的运维机制会变化。一旦智能体被容器化并置于稳定接口之后，选择宿主就是部署决策，而不是重写代码。

> **端到端运行本笔记本的成本：** 约 **1–2 美元**的 Anthropic API 调用费用，加上**几分钱**的 Modal 计算费用。每个层级都有拆除步骤。

全部部署代码都位于本笔记本旁边的 [﷐0﷑](./hosting/) 目录中。

## 开始之前：你应该使用 Agent SDK 吗？

如果你在做**面向客户的聊天产品**，请先看 [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)。它开箱即用地提供托管、会话和界面，你可以跳过本笔记本的大部分内容。

当你需要**编程式控制**时，Agent SDK 才是正确选择：批次和任务型的智能体、内部工具、嵌入你自己后端的智能体，或者必须自己掌控基础设施的受监管环境。如果你属于这种情况，请继续读下去。

## 心智模型

有三个名词必须分清：

- **进程**是一个正在运行、已加载 SDK 的 Python 解释器。它与 Anthropic API 通信。
- **会话**是一段对话：提示词历史、工具使用和结果，SDK 把它们写入磁盘，以便你之后用 `resume=` 恢复。
- **容器**是打包好的进程加上它的文件系统：你的智能体代码、SDK、Node，以及存放会话的位置。

与进程内的 SDK（OpenAI Agents SDK、Google ADK）不同——在那里“智能体”是你在 Web 服务器内部实例化的一个对象——Claude Agent SDK 的智能体本身**就是**一个进程。这使得隔离变得简单（一个容器 = 一个影响范围），但也意味着托管是分布式系统问题，而不是 `pip install` 问题。

任何层级的任何部署，都必须完成同样四项工作：

```
┌────────────────────────────────────────────────────────────────────┐
│  caller ──► gateway ──► [ spawn | route ] ──► agent container ──► API
│                 │                                     │
│                 └── auth (not the agent's job)        └── /data  (persist)
│                                                                      │
│  orchestrator ──────────────── lifecycle ────────────────────────────┘
└────────────────────────────────────────────────────────────────────┘
```

1. **派生**容器：有任务到达时启动容器
2. **路由**每个请求，送到拥有该会话的容器
3. **生命周期**：终止空闲容器，重启崩溃的容器
4. **持久化**会话记录，使重启不会丢失对话

层级 1 全部手工完成这四项。层级 2 把派生和生命周期交给 Modal。层级 3 把四项全部交给 Kubernetes 加一个小型网关。智能体容器始终不变。

## 我们要部署的智能体

我们复用笔记本 00 中的 [﷐0﷑](./research_agent/agent.py)，即带 `WebSearch` 和 `Read` 的一句话研究智能体。如果你还没做笔记本 00，请先完成它；本笔记本假定该智能体已经可用。

`hosting/` 唯一新增的是一个薄薄的 HTTP 服务器和一个 Dockerfile。系统提示词直接来自 `research_agent.agent`；我们是导入它而不是复制它。有一处有意为之的差异：托管服务器把工具列表收窄为只有 `WebSearch`。服务器上没有上传通道，因此 `Read` 能够触及的文件只有其他会话的记录和容器自身的环境文件，而一次被提示注入的网页结果可能诱导智能体泄露它们。`server.py` 中的注释说明了完整理由。

In [1]:
from research_agent.agent import DEFAULT_MODEL, RESEARCH_SYSTEM_PROMPT

print(f"model: {DEFAULT_MODEL}")
print(RESEARCH_SYSTEM_PROMPT)

model: claude-opus-4-6
You are a research agent specialized in AI.

When providing research findings:
- Always include source URLs as citations
- Format citations as markdown links: [Source Title](URL)
- Group sources in a "Sources:" section at the end of your response


### 环境准备

创建 `hosting/.env` 并填入你的 API 密钥。该文件已被 git 忽略。

In [2]:
%%bash
test -f hosting/.env || cp hosting/.env.example hosting/.env
echo 'Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.'
grep -q '^ANTHROPIC_API_KEY=sk-ant-' hosting/.env \
  && ! grep -q 'your-key-here' hosting/.env \
  && echo '✅ key looks set'

Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.


✅ key looks set


---
## 层级 1a — 临时式：一次提示词，一个容器，结束

最简单的部署方式：一个容器从环境变量读取提示词，把智能体**运行一次**，打印结果，然后退出。没有服务器，没有会话，没有状态。


> **模型说明：** 托管层默认使用 `claude-sonnet-4-6`，这样你在完成本笔记本时测试请求的成本保持低廉。在 `hosting/.env` 中设置 `MODEL=claude-opus-4-6`，可与笔记本 00 的配置完全一致。

对很多真实工作来说这就够了：发票处理、夜间报告生成、批次翻译、一次性分析。如果你的智能体的工作是“接收输入、产出输出、停止”，那么本节之后的内容你都不需要。

[﷐0﷑](./hosting/Dockerfile) 把智能体、SDK 以及 SDK 底层驱动的 Claude Code CLI 一起打包。构建上下文是 `claude_agent_sdk/`（本目录），而不是 `hosting/`，因为镜像还需要 `research_agent/` 和 `utils/`：

In [3]:
%%bash
docker build -f hosting/Dockerfile -t research-agent . | tail -n 3

#0 building with "orbstack" instance using docker driver

#1 [internal] load build definition from D

ockerfile
#1 transferring dockerfile: 2.30kB done
#1 DONE 0.0s

#2 resolve image config for docker-i

mage://docker.io/docker/dockerfile:1


#2 DONE 0.6s



#3 docker-image://docker.io/docker/dockerfile:1@sha256:87999aa3d42bdc6bea60565083ee17e86d1f3339802f

...


#12 [5/9] RUN pip install --no-cache-dir -r hosting/requirements.txt
#12 CACHED

#13 [8/9] COPY hos

ting/server.py hosting/run_once.py hosting/entrypoint.sh ./hosting/
#13 CACHED

#14 [9/9] RUN chmod 

+x hosting/entrypoint.sh  && touch hosting/__init__.py
#14 CACHED

#15 exporting to image
#15 export

ing layers done
#15 writing image sha256:b77b20f557cef5e4b9ef01212f3ba3a0895ee97ff8e31f79d9e1dc0cfb7

414f5 done
#15 naming to docker.io/library/research-agent done
#15 DONE 0.0s


In [4]:
%%bash
docker run --rm --env-file hosting/.env \
  -e PROMPT='What is the Claude Agent SDK, in one paragraph?' \
  research-agent

🤖 Thinking...


🤖 Using: ToolSearch()


✓ Tool completed


🤖 Thinking...


🤖 Using: WebSearch()


...

sdk/overview)
- [Building Agents with the Claude Agent SDK – Anthropic Engineering](https://www.an

thropic.com/engineering/building-agents-with-the-claude-agent-sdk)
- [Agent SDK Overview – Anthrop

ic API Docs](https://docs.anthropic.com/en/docs/claude-code/sdk/sdk-overview)
- [claude-agent-sdk-py

thon – GitHub](https://github.com/anthropics/claude-agent-sdk-python)
- [claude-agent-sdk-typescri

pt – GitHub](https://github.com/anthropics/claude-agent-sdk-typescript)


就这样。`entrypoint.sh` 没有看到 `serve` 参数，因此 [﷐2﷑](./hosting/run_once.py) 用 `$PROMPT` 调用 `research_agent.agent.send_query()`，然后以 0 退出。

**何时这就足够：** 每次调用都相互独立的任务型工作。可以从 cron、队列工作器、CI 步骤，或任何你运行 CLI 的地方启动它。

---
## 层级 1b — 混合式：加入服务器，使对话可以延续

临时模式无法保持对话；每次 `docker run` 都开启一个全新会话。对于聊天形态的智能体，你需要一个长期运行的进程，接受后续追问，并在每次时恢复正确的会话。

[﷐0﷑](./hosting/server.py) 是一个约 100 行的 FastAPI 应用，只做这件事，别无其他。这个接口就是每个层级都遵循的契约：

```
GET  /health                            → 200 {"status": "ok"}
POST /sessions/{session_id}/messages    → 200 text/event-stream of SDK messages
```

`server.py` 中有两点值得注意：

- **服务器没有鉴权。** 文档字符串已明确说明这一点。鉴权是网关的职责（层级 3 会展示它的位置）。服务器只校验 `session_id` 的格式，并信任调用方。
- **它维护一个从你的 `session_id` 到 SDK 内部 ID 的小型映射。** SDK 会生成自己的会话 ID，你无法指定。服务器从第一轮对话的 `ResultMessage` 中获知 SDK 的 ID，并在后续追问时把它传给 `resume=`。该映射与记录一起持久化保存在 `/data` 下。

用 docker-compose 启动它，它同时把 `./sessions` 挂载到 `/data`，因此记录能在重启后保留：

In [5]:
%%bash
cd hosting/docker && docker compose up --build -d
sleep 3
curl -s http://localhost:8000/health

 Image research-agent Building 


#1 [internal] load local bake definitions


#1 reading from stdin 579B done
#1 DONE 0.0s



#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 2.30kB done


#2 DONE 0.0s

#3 resolve image config for docker-image://docker.io/docker/dockerfile:1


...

 Container docker-research-agent-1 Creating 


 Container docker-research-agent-1 Created 


 Container docker-research-agent-1 Starting 


 Container docker-research-agent-1 Started 


{"status":"ok"}

发送一个提示词并流式接收响应（`-N` 关闭 curl 的缓冲，因此事件到达时你就能看到）：

In [6]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"What are the three most interesting AI agent trends right now?"}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01LpCYD1mQgNAmiDYq55RHyC", "name": "WebSearch", "input": {"query": "AI agent trends 2026"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, "usage": {"input_tokens": 685, "cache_cr … [truncated]

[... 6 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "Great question! Based on the latest research and reports, here are the **three most interesting AI agent trends** right now in 2026:\n\n---\n\n## \ud83e\udd1d 1. Multi-Agent Systems & Orchestration\nThe era of the single, all-purpose AI agent is giving way to **coordinat … [truncated]

event: message
data: {"subtype": "s

现在向**同一个** `session_id` 发送后续追问。智能体记得第一轮对话，因为服务器恢复了该会话：

In [7]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Tell me more about the second one."}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01YU5m6b66Kh6GcSN1Kbv7zq", "name": "WebSearch", "input": {"query": "context engineering AI agents 2026 techniques best practices"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, … [truncated]

[... 5 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "## \ud83e\uddf1 Deep Dive: Context Engineering\n\nContext engineering has quickly become **the defining AI skill of 2026**. Here's a thorough breakdown:\n\n---\n\n### What Is It, Exactly?\n\nContext engineering is the discipline of **designing what information an AI mode … [truncated]

event: message
data: {"subtype": "su

重启容器，再发送*另一个*后续追问。由于卷挂载保留了 `/data`，对话得以延续：

In [8]:
%%bash
cd hosting/docker && docker compose restart && sleep 3
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Summarize what we have discussed so far."}'

 Container docker-research-agent-1 Restarting 


 Container docker-research-agent-1 Started 


event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app

", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Gl

ob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "NotebookEdit", "WebFetch", "TodoWrite", "WebS

...

oid context rot and token blowouts.\n- **RAG alone isn't enough** \u2014 77% of IT leaders agree RAG

 is insufficient for production AI on its own.\n- Proper context engineering can improve agent task 

completion rates dramatically (e.g., **83% \u2192 96%**).\n\n---\n\nWould you like to explore any of

 these topics further?", "structured_output": null, "type": "ResultMessage"}



event: done
data: 



In [9]:
%%bash
# Teardown tier 1
cd hosting/docker && docker compose down

 Container docker-research-agent-1 Stopping 


 Container docker-research-agent-1 Stopped 
 Container docker-research-agent-1 Removing 


 Container docker-research-agent-1 Removed 


 Network docker_default Removing 


 Network docker_default Removed 


---
## 层级 2 — Modal：同样的镜像，现在它是一个 URL

层级 1 运行在你的机器上。层级 2 通过 `modal.Sandbox` 把**同一个 Dockerfile** 运行在 [Modal](https://modal.com) 上，它给你一个公开的 HTTPS URL、缩零伸缩，并且无需管理服务器。

那个 URL 是*公开的*：任何拿到它的人都能消耗你的 API 预算。层级 1 和层级 3 假定前面有一个做鉴权的网关；层级 2 没有网关，因此 `modal_app.py` 会为每次部署生成一个 bearer 令牌，并通过 `AGENT_AUTH_TOKEN` 传入。`server.py` 仅在该环境变量被设置时才强制校验令牌，因此其他层级不受影响。

[﷐0﷑](./hosting/modal/modal_app.py) 很短，因为智能体的任何部分都没有变化：

```python
app = modal.App.lookup("research-agent-hosting", create_if_missing=True)
image = modal.Image.from_dockerfile("hosting/Dockerfile", context_dir=".")
auth_token = secrets.token_urlsafe(32)
sandbox = modal.Sandbox.create(
    "serve",  # appended to the image's ENTRYPOINT, like compose's `command:`
    app=app,
    image=image,
    secrets=[
        modal.Secret.from_name("anthropic"),
        modal.Secret.from_dict({"AGENT_AUTH_TOKEN": auth_token}),
    ],
    volumes={"/data": sessions_volume},
    encrypted_ports=[8000],
)
print(sandbox.tunnels()[8000].url)
```

持久化使用挂载在 `/data` 的 `modal.Volume`，与层级 1 使用同样的 `CLAUDE_CONFIG_DIR` 技巧。（如果你的工作负载有许多沙箱并发写入并遇到 Volume 提交语义问题，可换用 [﷐3﷑](https://code.claude.com/docs/en/agent-sdk/session-storage)；层级 3 和生产环境部署用的也是它。）

一次性准备，**在你的终端中**执行（`modal setup` 会打开浏览器，因此无法在笔记本单元格中运行）：

```bash
pip install modal
modal setup
```

然后创建 Modal 将作为 `ANTHROPIC_API_KEY` 注入的密钥：

In [10]:
%%bash
modal secret create anthropic ANTHROPIC_API_KEY="$(grep ANTHROPIC_API_KEY hosting/.env | cut -d= -f2)"

Created a new secret 'anthropic' with the key 'ANTHROPIC_API_KEY'



Use it in your Modal app:




@app.[38;2;248;248;242;48;

2;39;40;34mfunction(secret

s=[[38;2;248;248;242;4

...

                 
    os[

0m.getenv[38;2;248;248;242

;48;2;39;40;34m("ANTHROPIC

_API_KEY")[48;2;39;40

;34m                                              
                             

In [11]:
%%bash
python hosting/modal/modal_app.py | tee /tmp/modal_deploy.out
MODAL_URL=$(awk '/^url:/ {print $2}' /tmp/modal_deploy.out)
MODAL_TOKEN=$(awk '/^token:/ {print $2}' /tmp/modal_deploy.out)
{ echo "MODAL_URL=$MODAL_URL"; echo "MODAL_TOKEN=$MODAL_TOKEN"; } > /tmp/modal_url.env

sandbox: sb-7R7zQ7TtX0h9eKZ8qslvwo
url:     https://ta-01ks91e217n9fymaxjtdc9k5bh-8000-kn9n102ljd7y4

majwav00kwg0.w.modal.host
token:   sb-…redacted…

⚠️  The URL is p

ublic. The token is the only thing gating it — don't share both.

Try it:
  curl -N -X POST https:

//ta-01ks91e217n9fymaxjtdc9k5bh-8000-kn9n102ljd7y4majwav00kwg0.w.modal.host/sessions/demo-1/messages

 \
    -H 'Authorization: Bearer sb-…redacted…' \
    -H 'Content-Type

: application/json' \
    -d '{"prompt":"What are the latest AI agent trends?"}'


In [12]:
%%bash
source /tmp/modal_url.env
curl -N -s -X POST "$MODAL_URL/sessions/demo-1/messages" \
  -H "Authorization: Bearer $MODAL_TOKEN" \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Give me a one-sentence summary of the Claude Agent SDK."}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "1566ffe4-2b20-4a68-82ff-283984b64451", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01PPS8yzMBzMnRhG2Hnpk5VL", "name": "WebSearch", "input": {"query": "Claude Agent SDK Anthropic 2026"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, "usage": {"input_tokens": 120 … [truncated]

[... 5 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "The **Claude Agent SDK** is Anthropic's framework that gives developers programmatic access to the same tools, agent loop, and context management that power Claude Code \u2014 enabling the creation of AI agents that can autonomously read files, run commands, search the w … [truncated]

event: message
data: {"subtype": "s

同样的接口，同样的镜像，不同的宿主。没有调用时，Modal 把沙箱缩到零，你无需付费。

拆除资源，避免为闲置资源付费：

In [13]:
%%bash
python hosting/modal/teardown.py

terminating sandbox sb-7R7zQ7TtX0h9eKZ8qslvwo
deleted volume research-agent-sessions


---
## 层级 3 — Kubernetes：当你需要自己掌控基础设施

层级 3 适用于多租户生产环境、受监管环境，或任何你需要完全掌控网络、隔离和成本的场合。智能体镜像和接口仍然完全相同；新增的是它*外围*的机制：

- 前端有一个**网关**，对调用方鉴权，并且只转发它们自己拥有的 `session_id`。`server.py` 省略的鉴权正是在这里完成。
- **每个会话一个 Pod** 的路由（网关 → Redis → pod），使每段对话拥有自己的影响范围。
- 一个由预热 pod 组成的**待机池**，使新会话不必承担冷启动时延。
- **出口封锁**（NetworkPolicy 加白名单代理），使被提示注入的智能体只能访问 `api.anthropic.com`，无法访问其他任何地方。

完整的清单、网关和逐步的架构讲解位于 [﷐0﷑](./hosting/kubernetes/)。它无需云账号，可在本地 [kind](https://kind.sigs.k8s.io/) 集群上端到端运行：

```bash
cd hosting/kubernetes
ANTHROPIC_API_KEY=sk-ant-... ./kind-quickstart.sh
```

快速上手脚本会为两个演示租户（`alice` 和 `bob`）打印 bearer 令牌。网关把每个会话限定给创建它的租户：与层级 1 和 2 相同的 `curl -N POST /sessions/{id}/messages`，现在指向 `:8080` 上的网关并带上 `Authorization：****** 头，对 alice 可用，而用 bob 的令牌发出同一请求会得到 403。README 的 *部署到你自己的集群* 一节说明真实集群所需的镜像仓库/ingress 改动。

*Kubernetes 层级建立在 Joe Shamon 和 Ben Lehrburger 的内部工作之上。*


---
## 使其达到生产可用

有两项生产关注点，各用几行代码即可接入。下面的单元格展示代码；[hosting 文档](https://code.claude.com/docs/en/agent-sdk/hosting) 给出完整的生产检查清单（鉴权、优雅关闭、空闲超时调优、自动伸缩、成本控制）。

### 可观测性

SDK 为每一轮对话和每次工具调用发出 OpenTelemetry 跨度。用两个环境变量把它指向你的采集器，`server.py` 无需任何代码改动（[文档](https://code.claude.com/docs/en/agent-sdk/observability)）：

In [14]:
# In docker-compose.yml / modal_app.py / your k8s Deployment:
#   OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
#   OTEL_SERVICE_NAME=research-agent

### 存活检查

`GET /health` 已经在 `server.py` 中。把编排器的存活探针指向它（compose 的 `healthcheck:`、Modal 健康检查、k8s 的 `livenessProbe`）。

### 超越卷的持久化

对单主机和 Modal 而言，`/data` 卷挂载已经足够。对多主机生产环境，使用把记录镜像到共享存储（S3、Postgres、Redis）的 [﷐1﷑ 适配器](https://code.claude.com/docs/en/agent-sdk/session-storage)。注意 SessionStore 只是镜像；本地磁盘写入总是先发生，镜像失败会发出 `mirror_error`，而不会中断智能体。

### 传输格式

`server.py` 流式输出原始的 SDK 消息类型。对 Cookbook 来说这没问题；对真实 API，你应定义稳定的传输格式，使 SDK 版本升级不会破坏客户端。

---
## 选择你的层级

| 层级 | 它给你什么 | 何时选择它 | 何时升级 |
|---|---|---|---|
| **1. Docker** | 一台由你掌控的机器上的容器。Compose 会重启它，bind mount 保留 `/data`。你只需运维一个 Docker 宿主。| 开发循环、内部工具、单租户应用、cron/批次任务。你可以手工重启它，且网络之外没有人调用它。| 那台机器之外的人需要一个 URL，或者“手工重启”不再可接受。|
| **2. Modal** | 同一个镜像置于公开 HTTPS URL 之后，带缩零伸缩和远程构建。你什么都不用运维。| 你现在就需要一个可调用的端点，流量呈突发状或大部分时间为零，并且每次部署的 bearer 令牌足以鉴权。| 你需要真正的多租户隔离、网络层的出口控制，或者你的平台团队要求工作负载跑在他们的集群上。|
| **3. Kubernetes** | 每个会话一个 Pod 的隔离、带租户限定会话的鉴权网关、出口封锁，以及预热待机池。你运维集群、网关和 Redis。| 多租户生产环境、受监管环境，或者你已经在运行 Kubernetes 并希望智能体遵循与其他服务相同的运维模式。| 这是本笔记本阶梯的顶层；从这里开始你调优自动伸缩、多区域路由和持久会话存储，而不是迁移。|

[hosting 指南](https://code.claude.com/docs/en/agent-sdk/hosting)的部署模式与这些层级直接对应。**临时会话**（模式 1：一次提示词、一个进程、退出）就是层级 1a：没有服务器，从 cron 或队列工作器在任意宿主上运行容器。**长期运行**和**混合会话**（模式 2 和 3，由一个进程跨轮次保持或重建对话状态）正是 `server.py` 用 `resume=` 加 `/data` 卷实现的方式；层级 1b、2 和 3 都服务于这种形态，区别只在于由谁维持该进程存活、调用方如何抵达它。**单容器**（模式 4，多个会话复用到一个容器中）正是层级 1b 和 2 所做的；当这种做法不再可接受、每个会话需要自己的影响范围时，层级 3 就派上用场。


---
## 附录 — 迁移到其他服务商

同样的 `hosting/Dockerfile`，只是部署命令不同。这些平台都暴露 8000 端口并给你一个 URL；在 `/data` 挂载某物以实现持久化。

**Fly Machines**
```bash
fly launch --dockerfile hosting/Dockerfile --no-deploy  # run from claude_agent_sdk/
fly volumes create data --size 1
fly deploy
```

**E2B**
```python
from e2b import Sandbox
sbx = Sandbox(template="research-agent")  # template built from hosting/Dockerfile
sbx.commands.run("./hosting/entrypoint.sh serve", background=True)
url = sbx.get_host(8000)
```

**Daytona**
```python
from daytona import Daytona, CreateSandboxFromImageParams
sbx = Daytona().create(CreateSandboxFromImageParams(image="research-agent"))
sbx.process.exec("./hosting/entrypoint.sh serve")
```

**Cloudflare Containers**
```ts
// wrangler.toml points at hosting/Dockerfile
export class Agent extends Container { defaultPort = 8000 }
```

**Vercel Sandbox**
```ts
import { Sandbox } from "@vercel/sandbox";
const sbx = await Sandbox.create({ image: "research-agent", ports: [8000] });
await sbx.runCommand({ cmd: "./hosting/entrypoint.sh", args: ["serve"], detached: true });
```